# Data Preprocessing Notes

Loading the Chicago census / crime / schools, automobile and laptop datasets, then formatting, type fixing, normalization (simple, min-max, z-score), binning and encoding.


# Chicago Crime Data — SQL Analysis
Load `ChicagoCrimeData.csv` into a SQLite database and answer key questions using SQL queries.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import sqlite3
import os
import numpy as np

## 2. Load All Three Datasets

In [2]:
import os

# Locate data files (works whether cwd is workspace root or notebook folder)
notebook_dir = r"c:\Users\USUARIO\OneDrive - INSTITUCION UNIVERSITARIA SALAZAR Y HERRERA\Escritorio\knowledge\01_sql"
for candidate in [os.getcwd(), os.path.join(os.getcwd(), "01_sql"), notebook_dir]:
    if os.path.isfile(os.path.join(candidate, "ChicagoCrimeData.csv")):
        data_dir = candidate
        break
else:
    data_dir = notebook_dir  # fallback to absolute path

print(f"Data directory: {data_dir}")

df_census  = pd.read_csv(os.path.join(data_dir, "ChicagoCensusData.csv"))
df_crime   = pd.read_csv(os.path.join(data_dir, "ChicagoCrimeData.csv"))
df_schools = pd.read_csv(os.path.join(data_dir, "ChicagoPublicSchools.csv"))
df_import  = pd.read_csv(os.path.join(data_dir, "imports-85.csv"))
df_laptops = pd.read_csv(os.path.join(data_dir, "laptops_train.csv"))

print(f"Census rows   : {len(df_census):,}")
print(f"Crime rows    : {len(df_crime):,}")
print(f"Schools rows  : {len(df_schools):,}")
print(f"Import rows   : {len(df_import):,}")
print(f"Laptops rows  : {len(df_laptops):,}")


Data directory: c:\Users\USUARIO\OneDrive - INSTITUCION UNIVERSITARIA SALAZAR Y HERRERA\Escritorio\knowledge\01_sql
Census rows   : 78
Crime rows    : 533
Schools rows  : 566
Import rows   : 205
Laptops rows  : 977


In [3]:
df_census.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
COMMUNITY_AREA_NUMBER,77.0,NaN,NaN,NaN,39.0,22.371857,1.0,20.0,39.0,58.0,77.0
COMMUNITY_AREA_NAME,78,78,Rogers Park,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PERCENT_OF_HOUSING_CROWDED,78.0,NaN,NaN,NaN,4.920513,3.658981,0.3,2.325,3.85,6.8,15.8
PERCENT_HOUSEHOLDS_BELOW_POVERTY,78.0,NaN,NaN,NaN,21.739744,11.457231,3.3,13.35,19.05,29.15,56.5
PERCENT_AGED_16__UNEMPLOYED,78.0,NaN,NaN,NaN,15.341026,7.499497,4.7,9.2,13.85,20.0,35.9
PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,78.0,NaN,NaN,NaN,20.330769,11.746514,2.5,12.075,18.65,26.6,54.8
PERCENT_AGED_UNDER_18_OR_OVER_64,78.0,NaN,NaN,NaN,35.717949,7.284421,13.5,32.15,38.05,40.5,51.5
PER_CAPITA_INCOME,78.0,NaN,NaN,NaN,25597.0,15196.405541,8201.0,15804.75,21668.5,28715.75,88669.0
HARDSHIP_INDEX,77.0,NaN,NaN,NaN,49.506494,28.690556,1.0,25.0,50.0,74.0,98.0


## PREPROCESSING DATA  IN PYTHON 


In [4]:
# colums modificacions
df_census["COMMUNITY_AREA_NUMBER"]= df_census["COMMUNITY_AREA_NUMBER"]+1

#show the first 10 rows of the census dataframe
df_census.head(10)

#drop misisng values 
df_census.dropna(inplace=True)

#mising values nan 
df_census.isna().sum()

#replacing data its better than missing values . average value of the columns 
df_census.fillna(df_census.mean(numeric_only=True), inplace=True)






,COMMUNITY_AREA_NUMBER,COMMUNITY_AREA_NAME,PERCENT_OF_HOUSING_CROWDED,PERCENT_HOUSEHOLDS_BELOW_POVERTY,PERCENT_AGED_16__UNEMPLOYED,PERCENT_AGED_25__WITHOUT_HIGH_SCHOOL_DIPLOMA,PERCENT_AGED_UNDER_18_OR_OVER_64,PER_CAPITA_INCOME,HARDSHIP_INDEX
0,2.0,Rogers Park,7.7,23.6,8.7,18.2,27.5,23939,39.0
1,3.0,West Ridge,7.8,17.2,8.8,20.8,38.5,23040,46.0
2,4.0,Uptown,3.8,24.0,8.9,11.8,22.2,35787,20.0
3,5.0,Lincoln Square,3.4,10.9,8.2,13.4,25.5,37524,17.0
4,6.0,North Center,0.3,7.5,5.2,4.5,26.2,57123,6.0
...,...,...,...,...,...,...,...,...,...
72,74.0,Washington Height,1.1,16.9,20.8,13.7,42.6,19713,48.0
73,75.0,Mount Greenwood,1.0,3.4,8.7,4.3,36.8,34381,16.0
74,76.0,Morgan Park,0.8,13.2,15.0,10.8,40.3,27149,30.0
75,77.0,O'Hare,3.6,15.4,7.1,10.9,30.3,25828,24.0


## DATA FORMATTING 
ITS A STANDARD OF REPRESENTATIONS ....

In [5]:
df_import["city-mpg"] = 235 / pd.to_numeric(
    df_import["city-mpg"], errors="coerce"
)
df.rename(columns={"city-mpg": "city-l/100km"}, inplace=True)

NameError: name 'df' is not defined

## wrong datatype

In [ ]:
df_census.dtypes
df_census["COMMUNITY_AREA_NUMBER"] = df_census["COMMUNITY_AREA_NUMBER"].astype("int64")
df_census.dtypes

## HEADERS CONFIG

In [ ]:
headers = ["Manufacturer", "Model_Name", "Category", "Screen_Size", "Screen", "CPU", "RAM", "Storage", "GPU", "OS", "OS_Version", "Weight_kg", "Price"]
df_laptops.columns = headers
print(df_laptops.head(10))


## DATA NORMALIZATION

simple feature scaling 

In [ ]:

# Para df_laptops
df_laptops_scaled = df_laptops.copy()
for col in df_laptops.select_dtypes(include=['float64', 'int64']).columns:
    x_max = df_laptops[col].max()
    x_min = df_laptops[col].min()
    if x_max != x_min:  # Evitar división por cero
        df_laptops_scaled[col] = (df_laptops[col] - x_min) / (x_max - x_min)

print("Laptops scaled (0-1 range):")
print(df_laptops_scaled.head(10))
print("\nDtypes after scaling:")
print(df_laptops_scaled.dtypes)

# Para df_import
df_import_scaled = df_import.copy()
for col in df_import.select_dtypes(include=['float64', 'int64']).columns:
    x_max = df_import[col].max()
    x_min = df_import[col].min()
    if x_max != x_min:
        df_import_scaled[col] = (df_import[col] - x_min) / (x_max - x_min)

print("\n\nImport data scaled (0-1 range):")
print(df_import_scaled.head(10))
print("\nDtypes after scaling:")
print(df_import_scaled.dtypes)

min-max scaling 

In [ ]:
## DATA NORMALIZATION - Min-Max Scaling

# Formula: x_new = (x - x_min) / (x_max - x_min)

# Para df_laptops
df_laptops_scaled = df_laptops.copy()
numeric_cols = df_laptops.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:
    x_min = df_laptops[col].min()
    x_max = df_laptops[col].max()
    if x_max != x_min:  # Evitar división por cero
        df_laptops_scaled[col] = (df_laptops[col] - x_min) / (x_max - x_min)

print("✓ Laptops escalados (rango 0-1):")
print(df_laptops_scaled.head(10))
print(f"\nRango de valores: {df_laptops_scaled[numeric_cols].min().min():.2f} a {df_laptops_scaled[numeric_cols].max().max():.2f}")

# Para df_import
df_import_scaled = df_import.copy()
numeric_cols = df_import.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:
    x_min = df_import[col].min()
    x_max = df_import[col].max()
    if x_max != x_min:
        df_import_scaled[col] = (df_import[col] - x_min) / (x_max - x_min)

print("\n✓ Import data escalados (rango 0-1):")
print(df_import_scaled.head(10))
print(f"\nRango de valores: {df_import_scaled[numeric_cols].min().min():.2f} a {df_import_scaled[numeric_cols].max().max():.2f}")

z-score 

In [ ]:
## DATA NORMALIZATION - Z-Score (Standardization)

# Formula: z = (x - mean) / std_dev
# Resultado: media = 0, desviación estándar = 1

# Para df_laptops
df_laptops_zscore = df_laptops.copy()
numeric_cols = df_laptops.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:
    mean = df_laptops[col].mean()
    std = df_laptops[col].std()
    if std != 0:  # Evitar división por cero
        df_laptops_zscore[col] = (df_laptops[col] - mean) / std

print("✓ Laptops Z-Score normalizados:")
print(df_laptops_zscore.head(10))
print(f"\nMedia: {df_laptops_zscore[numeric_cols].mean().mean():.6f}")
print(f"Desv. Estándar: {df_laptops_zscore[numeric_cols].std().mean():.6f}")

# Para df_import
df_import_zscore = df_import.copy()
numeric_cols = df_import.select_dtypes(include=['float64', 'int64']).columns

for col in numeric_cols:
    mean = df_import[col].mean()
    std = df_import[col].std()
    if std != 0:
        df_import_zscore[col] = (df_import[col] - mean) / std

print("\n✓ Import data Z-Score normalizados:")
print(df_import_zscore.head(10))
print(f"\nMedia: {df_import_zscore[numeric_cols].mean().mean():.6f}")
print(f"Desv. Estándar: {df_import_zscore[numeric_cols].std().mean():.6f}")

## BINNING  IN PYTHON 
converts numeric into categorical variables

In [ ]:
## BINNING IN PYTHON
# Converts numeric into categorical variables

# Opción 1: Binning con pandas.cut() - Intervalos iguales
print("=" * 60)
print("BINNING EXAMPLE 1: Equal-width bins (pandas.cut)")
print("=" * 60)

# Binning de Price en df_laptops
df_laptops_binned = df_laptops.copy()
df_laptops_binned["Price_Category"] = pd.cut(
    df_laptops["Price"],
    bins=4,
    labels=["Económica", "Media", "Premium", "Lujo"]
)

print("\n✓ Laptops con categorías de precio:")
print(df_laptops_binned[["Price", "Price_Category"]].head(15))
print(f"\nDistribución de categorías:")
print(df_laptops_binned["Price_Category"].value_counts().sort_index())

# Binning de horsepower en df_import
df_import_binned = df_import.copy()
# Convert horsepower to numeric first (handles strings like '?')
hp_numeric = pd.to_numeric(df_import["horsepower"], errors="coerce")

df_import_binned["Horsepower_Category"] = pd.cut(
    hp_numeric,
    bins=3,
    labels=["Bajo", "Medio", "Alto"],
    include_lowest=True
)

# Optional: label invalid/missing horsepower values
df_import_binned["Horsepower_Category"] = (
    df_import_binned["Horsepower_Category"]
    .cat.add_categories("Desconocido")
    .fillna("Desconocido")
)

print("\n✓ Autos con categorías de potencia:")
print(df_import_binned[["horsepower", "Horsepower_Category"]].head(15))
print(f"\nDistribución de potencia:")
print(df_import_binned["Horsepower_Category"].value_counts())

# Opción 2: Binning con pandas.qcut() - Intervalos basados en quantiles
print("\n" + "=" * 60)
print("BINNING EXAMPLE 2: Quantile-based bins (pandas.qcut)")
print("=" * 60)

df_laptops_qbinned = df_laptops.copy()
df_laptops_qbinned["Price_Quartile"] = pd.qcut(
    df_laptops["Price"],
    q=4,
    labels=["Q1 (Más barato)", "Q2", "Q3", "Q4 (Más caro)"],
    duplicates="drop"
)

print("\n✓ Laptops por cuartiles de precio:")
print(df_laptops_qbinned[["Price", "Price_Quartile"]].head(15))
print(f"\nDistribución por cuartiles:")
print(df_laptops_qbinned["Price_Quartile"].value_counts().sort_index())

# Opción 3: Binning personalizado con valores específicos
print("\n" + "=" * 60)
print("BINNING EXAMPLE 3: Custom bins")
print("=" * 60)

df_import_custom = df_import.copy()
bins_custom = [0, 100, 150, 200, 300]
labels_custom = ["< 100 HP", "100-150 HP", "150-200 HP", "> 200 HP"]

df_import_custom["HP_Range"] = pd.cut(
    pd.to_numeric(df_import["horsepower"], errors="coerce"),
    bins=bins_custom,
    labels=labels_custom,
    include_lowest=True
)

print("\n✓ Autos con rangos personalizados de potencia:")
print(df_import_custom[["horsepower", "HP_Range"]].head(15))
print(f"\nDistribución de rangos:")
print(df_import_custom["HP_Range"].value_counts().sort_index())

## TURNING CATEGORICAL VARIABLES INTO QUANTITATIVE VARIABLES IN PYTHON

In [ ]:
## METHOD 1: One-Hot Encoding — pd.get_dummies()
# Converts each category into a separate binary (0/1) column

# --- df_laptops ---
cat_cols_laptops = df_laptops.select_dtypes(include=["object"]).columns.tolist()
print("Categorical columns in laptops:", cat_cols_laptops)

df_laptops_ohe = pd.get_dummies(df_laptops, columns=cat_cols_laptops, drop_first=True)
print(f"\nShape before: {df_laptops.shape}  →  after: {df_laptops_ohe.shape}")
print(df_laptops_ohe.head(3))

# --- df_import ---
cat_cols_import = df_import.select_dtypes(include=["object"]).columns.tolist()
print("\nCategorical columns in import:", cat_cols_import)

df_import_ohe = pd.get_dummies(df_import, columns=cat_cols_import, drop_first=True)
print(f"\nShape before: {df_import.shape}  →  after: {df_import_ohe.shape}")
print(df_import_ohe.head(3))


In [ ]:
## METHOD 2: Label Encoding — assigns an integer to each unique category
# Best for ordinal categories (Low < Medium < High) or tree-based models

class LabelEncoder:
    def fit(self, y):
        y = pd.Series(y).astype(str)
        self.classes_ = y.unique().tolist()
        self._map = {label: i for i, label in enumerate(self.classes_)}
        return self

    def transform(self, y):
        y = pd.Series(y).astype(str)
        return y.map(self._map).to_numpy()

    def fit_transform(self, y):
        return self.fit(y).transform(y)

le = LabelEncoder()

# --- df_laptops: encode each categorical column in-place ---
if "df_laptops" in globals():
    df_laptops_le = df_laptops.copy()
else:
    base_dir = globals().get("data_dir", globals().get("notebook_dir", os.getcwd()))
    df_laptops_le = pd.read_csv(os.path.join(base_dir, "laptops_train.csv"))
for col in df_laptops.select_dtypes(include=["object"]).columns:
    df_laptops_le[col] = le.fit_transform(df_laptops[col].astype(str))

print("Laptops — Label Encoded:")
print(df_laptops_le.head(5))
print(df_laptops_le.dtypes)

# --- df_import: same ---
df_import_le = df_import.copy()
for col in df_import.select_dtypes(include=["object"]).columns:
    df_import_le[col] = le.fit_transform(df_import[col].astype(str))

print("\nImport — Label Encoded:")
print(df_import_le.head(5))


In [ ]:
## METHOD 3: Ordinal Encoding — manual mapping for ordered categories
# Use when you know the natural order: Low=0, Medium=1, High=2

# Example: map fuel-type in df_import
fuel_map = {"gas": 0, "diesel": 1}
df_import_ord = df_import.copy()
df_import_ord["fuel-type-encoded"] = df_import_ord["fuel-type"].map(fuel_map)

print("fuel-type → fuel-type-encoded:")
print(df_import_ord[["fuel-type", "fuel-type-encoded"]].value_counts().reset_index())

# Example: map aspiration
asp_map = {"std": 0, "turbo": 1}
df_import_ord["aspiration-encoded"] = df_import_ord["aspiration"].map(asp_map)

print("\naspiration → aspiration-encoded:")
print(df_import_ord[["aspiration", "aspiration-encoded"]].value_counts().reset_index())

# Example: OS category in laptops (if present)
if "OS" in df_laptops.columns:
    os_categories = df_laptops["OS"].unique()
    os_map = {cat: i for i, cat in enumerate(sorted(os_categories))}
    df_laptops_ord = df_laptops.copy()
    df_laptops_ord["OS-encoded"] = df_laptops_ord["OS"].map(os_map)
    print("\nOS → OS-encoded mapping:", os_map)
    print(df_laptops_ord[["OS", "OS-encoded"]].head(10))


## DATA WRANGLING 

In [ ]:
df_import["city-mpg"] = 235 / pd.to_numeric(df_import["city-mpg"], errors="coerce")
df_import.rename(columns={"city-mpg": "city-l/100km"}, inplace=True)

In [ ]:
# Replace missing values in df_import
df_import.replace("?", np.nan, inplace=True)

# Fill numeric columns with their mean
for col in df_import.select_dtypes(include=[np.number]).columns:
    df_import[col].fillna(df_import[col].astype(float).mean(), inplace=True)

print ("\n✓ df_import after handling missing values:")
  

In [ ]:
# Convert selected columns in the dataframe to a specific data type
df_import[["horsepower", "peak-rpm", "price"]] = df_import[
    ["horsepower", "peak-rpm", "price"]
].astype("float")

#data_type is int, float, char, etc. 

#data_type is int, float, char, etc. 

